In [0]:
%pip install databricks-feature-engineering

In [0]:
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient


class DemandFeatureStore:
    """Manages feature store operations for 7-day demand forecasting."""

    def __init__(self):
        self.source_table = "oil_gas_demo.gold.demand_features"
        self.feature_table = "oil_gas_demo.gold.demand_training_features"
        self.fe = FeatureEngineeringClient()
        self.primary_keys = ["forecast_date", "product_name", "destination_city"]

    def load_data(self):
        return spark.table(self.source_table)

    def prepare_training_data(self):
        """Self-join demand_features to add target_demand (actual demand 7 days ahead)."""
        features_df = self.load_data()

        features_with_target_date = features_df.withColumn(
            "target_date",
            F.date_add(F.col("transaction_date"), 7)
        )

        target_dataset = (
            features_with_target_date.alias("f")
            .join(
                features_df.alias("a"),
                (
                    (F.col("f.target_date") == F.col("a.transaction_date")) &
                    (F.col("f.product_name") == F.col("a.product_name")) &
                    (F.col("f.destination_city") == F.col("a.destination_city"))
                ),
                "left"
            )
            .select(
                F.col("f.transaction_date").alias("forecast_date"),
                F.col("f.target_date"),
                F.col("f.product_name"),
                F.col("f.destination_city"),
                F.col("f.total_demand"),
                F.col("f.avg_unit_price"),
                F.col("f.total_inventory"),
                F.col("f.day_of_week"),
                F.col("f.month"),
                F.col("f.demand_lag_1"),
                F.col("f.demand_lag_7"),
                F.col("f.rolling_avg_7"),
                F.col("a.total_demand").alias("target_demand")
            )
        )
        return target_dataset

    def get_training_dataset(self):
        """Return rows with a known target (non-null target_demand)."""
        return self.prepare_training_data().filter(F.col("target_demand").isNotNull())

    def check_duplicates(self):
        """Check for duplicate primary keys in the training dataset."""
        return (
            self.get_training_dataset()
            .groupBy(*self.primary_keys)
            .count()
            .filter(F.col("count") > 1)
        )

    def create_feature_table(self):
        """Create the Feature Store table with training data."""
        training_dataset = self.get_training_dataset()
        self.fe.create_table(
            name=self.feature_table,
            primary_keys=self.primary_keys,
            df=training_dataset,
            description="Demand forecasting features with actual demand 7 days ahead as training target.",
            tags={
                "project": "oil_gas_demand_forecasting",
                "model": "random_forest",
                "forecast_horizon": "7_days"
            }
        )
        print("Feature Store table created!")

    def update_feature_table(self):
        """Merge new data into the existing Feature Store table."""
        training_dataset = self.get_training_dataset()
        self.fe.write_table(
            name=self.feature_table,
            df=training_dataset,
            mode="merge"
        )
        print("Feature Store table updated!")

    def verify_data(self):
        """Read back and display the saved feature table."""
        df = spark.table(self.feature_table)
        print("Total rows:", df.count())
        display(df.orderBy("forecast_date").limit(10))


# Instantiate and load data
fs = DemandFeatureStore()
features_df = fs.load_data()

print("Demand features rows:", features_df.count())
print("\nDemand Features Schema:")
features_df.printSchema()

In [0]:
# Prepare training data via self-join (features from date X -> demand from date X+7)
target_dataset = fs.prepare_training_data()

display(target_dataset.limit(20))

# Count matched vs unmatched target rows
target_dataset.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("target_demand").isNotNull(), 1).otherwise(0)).alias("matched_rows"),
    F.sum(F.when(F.col("target_demand").isNull(), 1).otherwise(0)).alias("unmatched_rows")
).show()

In [0]:
# Keep only rows with a known actual target
training_dataset = fs.get_training_dataset()

print("Rows available for modeling:", training_dataset.count())
display(training_dataset.limit(10))

In [0]:
# Check for duplicate primary keys
duplicate_keys = fs.check_duplicates()
print("Duplicate keys:", duplicate_keys.count())
display(duplicate_keys.limit(10))

In [0]:
# Create the Feature Store table
fs.create_feature_table()

In [0]:
# Verify saved data
fs.verify_data()